In [1]:
import pandas as pd
from pathlib import Path

from nih.paths import ensure_dirs, CORPUS_DIR
from nih.io import list_zips_by_fy, join_projects_abstracts_fy
from nih.paths import PROJECTS_ZIPS_DIR, ABSTRACTS_ZIPS_DIR

ensure_dirs()

In [2]:
proj = list_zips_by_fy(PROJECTS_ZIPS_DIR)
abst = list_zips_by_fy(ABSTRACTS_ZIPS_DIR)
fys = sorted(set(proj).intersection(abst))
print("FYs available in BOTH:", fys[0], "->", fys[-1], "count:", len(fys))

FYs available in BOTH: 2005 -> 2024 count: 20


In [3]:
FY_START = fys[-5]
FY_END = fys[-1]

selected_fys = [fy for fy in fys if FY_START <= fy <= FY_END]
print("Selected FYs:", selected_fys)

Selected FYs: [2020, 2021, 2022, 2023, 2024]


In [4]:
KEEP_COLS = [
  "APPLICATION_ID", "FY", "IC_NAME", "ACTIVITY", "PROJECT_TITLE", "ABSTRACT_TEXT", "PROJECT_TERMS", "NIH_SPENDING_CATS", "TOTAL_COST",
]
def make_text(title, abstract):
  title = "" if pd.isna(title) else str(title).strip()
  abstract = "" if pd.isna(abstract) else str(abstract).strip()
  if title and abstract:
    return f"{title}\n\n{abstract}"
  return title or abstract

In [10]:
chunks = []
failures = []
for fy in selected_fys:
  try:
    df = join_projects_abstracts_fy(fy)
    cols = [c for c in KEEP_COLS if c in df.columns]
    df = df[cols].copy()
    df["text"] = [make_text(t, a) for t, a in zip(df.get("PROJECT_TITLE"), df.get("ABSTRACT_TEXT"))]
    chunks.append(df)
    missing_rate = df["ABSTRACT_TEXT"].isna().mean()
    missing_n = df["ABSTRACT_TEXT"].isna().sum()
    print(f"FY{fy}: rows={len(df):,} "
          f"missing_abs={missing_rate:.2%} "
          f"({missing_n:,})"
    )
  except Exception as e:
    failures.append((fy, repr(e)))
    print(f"FY{fy} FAILED: {e}")

FY2020: rows=82,428 missing_abs=5.57% (4,593)
FY2021: rows=82,940 missing_abs=1.93% (1,602)
FY2022: rows=83,877 missing_abs=0.88% (737)
FY2023: rows=85,024 missing_abs=1.11% (942)
FY2024: rows=83,314 missing_abs=6.87% (5,721)


In [6]:
corpus = pd.concat(chunks, ignore_index=True)
print("Total rows:", len(corpus))
print("Missing abstracts:", corpus["ABSTRACT_TEXT"].isna().mean())
print("Empty text:", (corpus["text"].fillna("").str.len() == 0).mean())

corpus.head(3)

Total rows: 417583
Missing abstracts: 0.03255640196080779
Empty text: 2.394733502082221e-06


,APPLICATION_ID,FY,IC_NAME,ACTIVITY,PROJECT_TITLE,ABSTRACT_TEXT,PROJECT_TERMS,NIH_SPENDING_CATS,TOTAL_COST,text
0,8676197,2020,Veterans Affairs,IK2,Improving Care for PTSD,DESCRIPTION (provided by applicant): BACK...,Affect;Area;Award;career;career development;Ca...,NaN,NaN,Improving Care for PTSD\n\nDESCRIPTION (provid...
1,8785548,2020,Veterans Affairs,I01,Design and Evaluation of User Centered Electro...,DESCRIPTION (provided by applicant): To d...,acronyms;Address;Affect;Archives;Area;Automati...,NaN,NaN,Design and Evaluation of User Centered Electro...
2,8867710,2020,Veterans Affairs,I01,Comparative Effectiveness of Delivery Methods ...,? DESCRIPTION (provided by applicant): ...,18 year old;Address;Adult;Adult Children;Age;A...,NaN,NaN,Comparative Effectiveness of Delivery Methods ...


In [7]:
corpus_model = corpus.dropna(subset=["ABSTRACT_TEXT"]).copy()
corpus_model = corpus_model[corpus_model["text"].fillna("").str.len() > 0]
print("Modelable rows:", len(corpus_model))

Modelable rows: 403988


In [8]:
out_path = CORPUS_DIR / "corpus.parquet"
corpus_model.to_parquet(out_path, index=False)
print("Wrote:", out_path)

Wrote: D:\nih_data\processed\corpus\corpus.parquet


In [9]:
if failures:
  fail_df = pd.DataFrame(failures, columns=["FY", "error"])
  fail_path = CORPUS_DIR / "build_failures.csv"
  fail_df.to_csv(fail_path, index=False)
  print("Failures saved to:", fail_path)
else:
  print("No failures!")

No failures!
